# BoTorch Acquisition Functions

This tutorial shows how to use the BoTorch acquisition function wrappers from
[`alf_tools.optimizer.acquisition_functions.botorch_acqs`](https://instadeepai.github.io/alf/api/alf_tools/optimizer/acquisition_functions/botorch_acqs/).

These wrappers accept **either** a native BoTorch `Model` **or** an ALF `BaseModel`
and insert a `BoTorchModelAdapter` automatically when needed. Here we use ALF's
`GPModel` as the surrogate to demonstrate the full integration.

Three usage patterns are covered:

| Pattern | When to use |
|---------|-------------|
| **Functional API** (`expected_improvement`, `upper_confidence_bound`, …) | Quick scripting; accepts a tensor, returns `Predictions` |
| **`BotorchAcquisitionFunction`** | Hydra-driven configs; implements the ALF `AcquisitionFunction` interface |
| **`@acquisition` decorator** | Custom BoTorch factories with automatic model adaptation |

### Environment Setup

From the `/tutorials` directory:
```
uv sync
source .venv/bin/activate
```
Select `.venv` as the kernel when prompted.

## 1. Imports

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from alf_core.dataclasses.candidate import Candidate, Modality
from alf_core.dataclasses.labelled_candidates import LabelledCandidates
from alf_tools.models import FeaturizerConfig, GPModel, GPModelConfig, GPTrainConfig
from alf_tools.optimizer.acquisition_functions.botorch_acqs import (
    BotorchAcquisitionConfig,
    BotorchAcquisitionFunction,
    acquisition,
    expected_improvement,
    log_noisy_expected_improvement,
    probability_of_improvement,
    upper_confidence_bound,
)
from alf_tools.utils.botorch_utils import candidates_to_tensor

print("Imports successful")

## 2. Train a GP Surrogate

We fit a `GPModel` to a synthetic 1D function.  The `"precomputed"` featuriser
accepts `Candidate.data` values that are already `np.ndarray` or `torch.Tensor`,
so no custom featuriser function is needed for tabular data.

In [ ]:
np.random.seed(42)
X_MIN, X_MAX = 0.0, 2 * np.pi
N_TRAIN = 20


def true_fn(x):
    return np.sin(x) + 0.5 * np.sin(3 * x)


x_train = np.sort(np.random.uniform(X_MIN, X_MAX, N_TRAIN))
y_train = true_fn(x_train) + np.random.randn(N_TRAIN) * 0.1


def make_candidates(x_array):
    """Wrap 1D points as ALF Candidates with precomputed numpy features."""
    return [Candidate(data=np.array([xi]), modality=Modality.TABULAR) for xi in x_array]


train_data = LabelledCandidates(make_candidates(x_train), y_train)
search_candidates = make_candidates(np.linspace(X_MIN, X_MAX, 100))

gp = GPModel(
    model_config=GPModelConfig(kernel_type="matern", matern_nu=2.5),
    train_config=GPTrainConfig(num_iterations=100, log_frequency=100),
    featurizer_config=FeaturizerConfig(featurizer_type="precomputed"),
)
gp.train(train_data)

best_f = float(y_train.max())
print(f"GP trained on {N_TRAIN} points")
print(f"Best observed value (best_f): {best_f:.3f}")
print(f"Learned hyperparameters: {gp.get_hyperparameters()}")

## 3. Functional API

Each factory (`expected_improvement`, `upper_confidence_bound`, …) returns an
`_AcquisitionCallable` that:
- accepts a **tensor** of shape `(n, d)`
- returns a `Predictions` object with acquisition scores as `.means`

When the model is an ALF `BaseModel`, the factory wraps it in a `BoTorchModelAdapter`
automatically — the caller never needs to do this manually.

The `.name` and `.kwargs` attributes on the returned callable are useful for
logging and config round-trips.

In [ ]:
# Convert ALF candidates to the tensor format expected by BoTorch
X = candidates_to_tensor(search_candidates)  # shape (100, 1)

# --- Expected Improvement ---
ei_acq = expected_improvement(gp, best_f=best_f)
ei_scores = ei_acq(X)  # returns Predictions

print("=== Expected Improvement ===")
print(f"  ei_acq.name:   {ei_acq.name}")
print(f"  ei_acq.kwargs: {ei_acq.kwargs}")
print(f"  scores shape:  {ei_scores.means.shape}")
print(f"  scores range:  [{ei_scores.means.min():.4f}, {ei_scores.means.max():.4f}]")

# --- Upper Confidence Bound ---
ucb_acq = upper_confidence_bound(gp, beta=2.0)
ucb_scores = ucb_acq(X)

print("\n=== Upper Confidence Bound ===")
print(f"  ucb_acq.name:   {ucb_acq.name}")
print(f"  ucb_acq.kwargs: {ucb_acq.kwargs}")
print(f"  scores shape:   {ucb_scores.means.shape}")
print(f"  scores range:   [{ucb_scores.means.min():.4f}, {ucb_scores.means.max():.4f}]")

EI is non-negative and peaks near unexplored regions where improvement over `best_f`
is likely.  UCB (`mean + β·std`) is not bounded below zero — it balances exploitation
(high mean) and exploration (high uncertainty) with `β` controlling the trade-off.

In [ ]:
preds = gp.predict(search_candidates)
x_search = np.linspace(X_MIN, X_MAX, 100)

fig, axes = plt.subplots(3, 1, figsize=(10, 9), sharex=True)

# Top: GP prediction
ax = axes[0]
ax.plot(x_search, true_fn(x_search), "k--", linewidth=1.2, alpha=0.6, label="True fn")
ax.scatter(x_train, y_train, s=50, color="black", zorder=5, label="Training data")
ax.plot(x_search, preds.means, color="#7b2d8b", linewidth=2, label="GP mean")
ax.fill_between(
    x_search,
    preds.means - 2 * np.sqrt(preds.variances),
    preds.means + 2 * np.sqrt(preds.variances),
    alpha=0.2,
    color="#7b2d8b",
    label="\u00b12\u03c3",
)
ax.axhline(best_f, color="green", linestyle=":", linewidth=1.5, label=f"best_f = {best_f:.2f}")
ax.set_ylabel("y")
ax.set_title("GP Surrogate (Matérn-2.5)")
ax.legend(fontsize=8, ncol=3)
ax.grid(True, alpha=0.3)

# Middle: EI
ax = axes[1]
ax.plot(x_search, ei_scores.means, color="#e74c3c", linewidth=2)
ax.fill_between(x_search, 0, ei_scores.means, alpha=0.2, color="#e74c3c")
ax.set_ylabel("EI score")
ax.set_title("Expected Improvement (non-negative; peaks where improvement is likely)")
ax.grid(True, alpha=0.3)

# Bottom: UCB
ax = axes[2]
ax.plot(x_search, ucb_scores.means, color="#2980b9", linewidth=2)
ax.fill_between(x_search, ucb_scores.means.min(), ucb_scores.means, alpha=0.2, color="#2980b9")
ax.set_ylabel("UCB score")
ax.set_xlabel("x")
ax.set_title("Upper Confidence Bound (\u03b2=2.0; balances mean and uncertainty)")
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Additional Factories

`probability_of_improvement` and `log_noisy_expected_improvement` follow the same
pattern.  The batch variant `log_noisy_expected_improvement` takes an `X_baseline`
tensor of already-observed points and is more robust to observation noise than
standard EI.

In [ ]:
# --- Probability of Improvement ---
poi_acq = probability_of_improvement(gp, best_f=best_f)
poi_scores = poi_acq(X)

print("=== Probability of Improvement ===")
print(f"  name:   {poi_acq.name}")
print(f"  kwargs: {poi_acq.kwargs}")
print(f"  scores in [{poi_scores.means.min():.4f}, {poi_scores.means.max():.4f}]")

# --- Log q-Noisy Expected Improvement ---
# X_baseline contains the training points; the acquisition integrates over them
# to produce noise-robust improvement estimates.
X_baseline = candidates_to_tensor(train_data.candidates)  # (n_train, 1)
lnei_acq = log_noisy_expected_improvement(gp, X_baseline=X_baseline)
lnei_scores = lnei_acq(X)

print("\n=== Log q-Noisy Expected Improvement ===")
print(f"  name:         {lnei_acq.name}")
print(f"  X_baseline shape: {X_baseline.shape}")
print(f"  scores in [{lnei_scores.means.min():.4f}, {lnei_scores.means.max():.4f}]")

## 5. `BotorchAcquisitionFunction` (Hydra-driven)

`BotorchAcquisitionFunction` implements the ALF `AcquisitionFunction` interface:
`__call__(candidates, state) → LabelledCandidates`.  It is configured via a
`BotorchAcquisitionConfig` that validates the `_target_` class.

The config accepts any class under `botorch.acquisition`.  At call time, the model
is injected from `state.surrogate.model` — it must **not** appear in the config.

> **In a real `DesignTask`**, `state` is the `State` object produced by `task.setup()`.  
> Below we use a minimal stub to keep the example self-contained.

In [ ]:
# Minimal stub that mirrors state.surrogate.model used in a real DesignTask
class _MockSurrogate:
    def __init__(self, model):
        self.model = model


class _MockState:
    def __init__(self, model):
        self.surrogate = _MockSurrogate(model)


state = _MockState(gp)

# ── Expected Improvement via Hydra config ──────────────────────────────────────
cfg = BotorchAcquisitionConfig({
    "_target_": "botorch.acquisition.analytic.ExpectedImprovement",
    "best_f": best_f,
})
ei_fn = BotorchAcquisitionFunction(cfg)
result = ei_fn(search_candidates, state)

print("=== BotorchAcquisitionFunction (EI) ===")
print(f"  Returns: {type(result).__name__}")
print(f"  Labels shape: {result.labels.shape}")
print(f"  Top-3 x positions: {[round(c.data[0], 3) for c in result.get_top_k(3).candidates]}")
print(f"  Top-3 scores:      {[round(s, 4) for s in result.get_top_k(3).labels.tolist()]}")

Switching acquisition functions is a one-line config change — no code changes elsewhere.

In [ ]:
# ── Upper Confidence Bound via Hydra config ────────────────────────────────────
ucb_cfg = BotorchAcquisitionConfig({
    "_target_": "botorch.acquisition.analytic.UpperConfidenceBound",
    "beta": 3.0,
})
ucb_fn = BotorchAcquisitionFunction(ucb_cfg)
ucb_result = ucb_fn(search_candidates, state)

print("=== BotorchAcquisitionFunction (UCB, beta=3.0) ===")
top3 = ucb_result.get_top_k(3)
print(f"  Top-3 x positions: {[round(c.data[0], 3) for c in top3.candidates]}")
print(f"  Top-3 scores:      {[round(s, 4) for s in top3.labels.tolist()]}")

## 6. Custom Factories with `@acquisition`

The `@acquisition` decorator handles model adaptation so your factory always
receives a proper `botorch.models.model.Model`.  When called with an ALF
`BaseModel`, the decorator wraps it in a `BoTorchModelAdapter` before passing it
to the function.  Positional and keyword arguments are captured in `.kwargs` for
config serialisation.

In [ ]:
from botorch.acquisition.analytic import UpperConfidenceBound as _UCB


@acquisition
def exploration_ucb(model, beta: float = 5.0):
    """UCB variant with a configurable exploration parameter."""
    return _UCB(model=model, beta=beta)


# Works directly with the ALF GPModel — no manual BoTorchModelAdapter needed
explore_acq = exploration_ucb(gp, beta=5.0)
explore_scores = explore_acq(X)

print("=== Custom @acquisition factory ===")
print(f"  name:   {explore_acq.name}")
print(f"  kwargs: {explore_acq.kwargs}")
print(f"  scores — mean: {explore_scores.means.mean():.4f}, max: {explore_scores.means.max():.4f}")

Higher `β` shifts attention towards high-uncertainty regions (exploration).  Here
we compare three values to see how the acquisition landscape changes.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

betas = [0.5, 2.0, 5.0]
colors = ["#2ecc71", "#e67e22", "#e74c3c"]

for beta, color in zip(betas, colors):
    acq = exploration_ucb(gp, beta=beta)
    scores = acq(X)
    ax.plot(x_search, scores.means, color=color, linewidth=2, label=f"\u03b2 = {beta}")

# Mark training locations on the x-axis
ax.scatter(
    x_train,
    np.full_like(x_train, ax.get_ylim()[0]),
    marker="|",
    s=100,
    color="black",
    label="Training data",
    zorder=5,
)
ax.set_xlabel("x")
ax.set_ylabel("UCB score")
ax.set_title("Effect of \u03b2 on UCB: Higher \u03b2 \u2192 More Exploration")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Low \u03b2: UCB mostly tracks the GP mean (exploitation).")
print("High \u03b2: UCB lifts scores in sparse, high-uncertainty regions (exploration).")

## Key Points

- **Model-agnostic**: All factories and `BotorchAcquisitionFunction` accept either a native
  BoTorch `Model` or an ALF `BaseModel` — `BoTorchModelAdapter` is inserted automatically.
- **Functional API** returns `Predictions` (tensor-in / `Predictions`-out); ideal for scripting
  and standalone evaluation.
- **`BotorchAcquisitionFunction`** implements the ALF `AcquisitionFunction` interface
  (`__call__(candidates, state) → LabelledCandidates`); use it wherever ALF expects an
  `AcquisitionFunction`.
- **`@acquisition` decorator**: wraps any BoTorch acquisition factory, stores kwargs for
  config serialisation via `.name` and `.kwargs`.
- **Variance required**: models must provide variance estimates. Deterministic models
  (e.g. `CNNModel`) cannot be used with analytic BoTorch acquisitions and will raise a
  `ValueError` with a descriptive message.
- **Available factories**: `expected_improvement`, `upper_confidence_bound`,
  `probability_of_improvement`, `log_noisy_expected_improvement`.

See `tools/alf_tools/optimizer/acquisition_functions/botorch_acqs.py` for the full
implementation and `tools/tests/optimizer/test_botorch_acqs.py` for more usage examples.